# Обучение ИИ-детектора (XLM-R → ONNX)
Среда: **Runtime → Change runtime type → GPU (T4)**.
Данные: положите реальные тексты в `data/human/` и `data/ai/<generator>/`,
либо используйте синтетический бутстрап (ячейка ниже).


## 1. Код проекта


In [ ]:
!git clone https://github.com/13unethical/antiplagiat.git || true
%cd antiplagiat/backend/ml
!pip -q install -r requirements.txt


## 2. Данные
Загрузите свой датасет (zip с папками human/ ai/) или оставьте синтетику для первого прогона.


In [ ]:
# Вариант A: загрузить архив с данными
# from google.colab import files; up=files.upload()  # data.zip со структурой human/ ai/<gen>/
# !unzip -o data.zip -d data

# Вариант B (быстрый старт): синтетический бутстрап
!python make_dataset.py --out data --synth 600
# Автогенерация ИИ-текстов (опц.): задайте ключ и раскомментируйте
# import os; os.environ['ANTHROPIC_API_KEY']='...'
# !python make_dataset.py --out data --gen-llm data/topics.txt


## 3. Обучение


In [ ]:
!python train.py --data data --model xlm-roberta-base --epochs 3 --out runs/ai-v1


## 4. Оценка (при наличии holdout.jsonl — честная LOMO-оценка)


In [ ]:
!python eval.py --model runs/ai-v1 --data data/val.jsonl


## 5. Экспорт в ONNX и скачивание


In [ ]:
!python export_onnx.py --model runs/ai-v1 --out export
!cd export && zip -r ../ai_model.zip . >/dev/null && echo OK
from google.colab import files; files.download('ai_model.zip')


## 6. Подключение к бэкенду
Распакуйте `ai_model.zip` в `backend/app/ml_model/`, в `.env` задайте
`AI_MODEL_DIR=./app/ml_model`, установите `onnxruntime transformers`, перезапустите backend.
